In [1]:
import os
import sys

project_root = os.path.abspath("..")
sys.path.append(project_root)

In [2]:
import pandas as pd

from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from Agents.retrieval_agent import RetrievalAgent, ProductQuery
from Agents.review_agent import ReviewAgent
from Agents.ranking_agent import RankingAgent

load_dotenv()

C:\Users\srush\AppData\Local\Temp\ipykernel_2252\3710450719.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

In [3]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [4]:
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_db = FAISS.load_local(
    "../Data/Cleaned/faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

In [5]:
reviews_df = pd.read_parquet(
    "../Data/Cleaned/reviews_sample.parquet"
)

In [6]:
retrieval_agent = RetrievalAgent(vector_db)

review_agent = ReviewAgent(
    llm,
    reviews_df
)

ranking_agent = RankingAgent(llm)

In [7]:
query = ProductQuery(
    product_type="phone",
    brand="Motorola",
    currency="USD",
    budget="30000",
    features=[
        "good battery life",
        "sound quality good"
    ]
)

In [8]:
retrieved_products = retrieval_agent.retrieve(query)

print(f"Retrieved {len(retrieved_products)} products")

Retrieved 4 products


In [9]:
products = []

for doc, retrieval_score in retrieved_products:

    review = review_agent.summarize_reviews(
        doc.metadata["parent_asin"]
    )

    products.append({
        "parent_asin": doc.metadata["parent_asin"],

        "title": doc.metadata["title"],

        "price": doc.metadata.get("price"),

        # Change this if your dataset uses another currency.
        # Use None when currency is unknown.
        "currency": doc.metadata.get("currency"),

        "retrieval_score": float(
            doc.metadata.get(
                "retrieval_score",
                retrieval_score
            )
        ),

        "average_rating": float(
            doc.metadata.get(
                "average_rating",
                0
            )
            or 0
        ),

        "rating_number": int(
            doc.metadata.get(
                "rating_number",
                0
            )
            or 0
        ),

        "overall_sentiment": review.get(
            "overall_sentiment",
            "unknown"
        ),

        "pros": review.get(
            "pros",
            []
        ),

        "cons": review.get(
            "cons",
            []
        ),

        "recommended_for": review.get(
            "recommended_for",
            ""
        ),

        "avoid_if": review.get(
            "avoid_if",
            ""
        ),

        "review_summary": review.get(
            "summary",
            ""
        )
    })

In [10]:
for p in products:
    print(
        p["title"],
        p["retrieval_score"]
    )

eMorevalue Earbuds Noise Cancelling Magnetic Earphone Headphones for Galaxy A14 5G / A13 5G / A13 / A23 / A12 / A51 / A52 5G / A71 / A03S (A- Black) 0.45126539723700404
Cellet Retractable Stereo in-Ear Headphone, Ear-Bud Compatible to Kyocera DuraForce Pro XV LTE Cadence LTE Pro 2 Asus ZenFone V Live Palm Phone Sonim XP5 XP8 XP5s RED Hydrogen One Essential Phone 0.44876710159323974
Moto G7+ Plus (64GB, 4GB) 6.2" FHD+ Max Vision, Snapdragon 636, IP54 Splash Resistant, 4G Volte (T-Mobile, Metro, Ultra) XT1965-T (Black) Renewed 0.42539856656756725
Premium Faux Leather Vertical Swivel Belt Clip Holster for Samsung Galaxy Mega 6.3 Mega 5.8 Nokia 1520 Lumia 1320（Black ）+White VanGoddy Headphone 0.375180046475823


In [11]:
ranking = ranking_agent.rank_products(
    query,
    products
)

In [12]:
for product in ranking.products:

    print("=" * 80)

    print("Rank:", product.rank)
    print("Title:", product.title)
    print("Average Rating:", product.average_rating)
    print("Retrieval Score:", product.retrieval_score)
    print("Final Score:", product.final_score)
    print("Reason:", product.reason)

    print()

Rank: 1
Title: eMorevalue Earbuds Noise Cancelling Magnetic Earphone Headphones for Galaxy A14 5G / A13 5G / A13 / A23 / A12 / A51 / A52 5G / A71 / A03S (A- Black)
Average Rating: 3.8
Retrieval Score: 0.45126539723700404
Final Score: 0.389475
Reason: This product has a retrieval score of 0.451, indicating relevance to the query. However, no customer-review evidence was available to verify features like battery life or sound quality.

Rank: 2
Title: Premium Faux Leather Vertical Swivel Belt Clip Holster for Samsung Galaxy Mega 6.3 Mega 5.8 Nokia 1520 Lumia 1320（Black ）+White VanGoddy Headphone
Average Rating: 1.9
Retrieval Score: 0.375180046475823
Final Score: 0.285132
Reason: With a low average rating of 1.9 and a retrieval score of 0.375, this product is less relevant. No customer-review evidence was available to confirm any requested features.

Rank: 3
Title: Cellet Retractable Stereo in-Ear Headphone, Ear-Bud Compatible to Kyocera DuraForce Pro XV LTE Cadence LTE Pro 2 Asus ZenFone 